# cluster03 bike_change 2차 피처 조합 비교

## 용어 설명

- `mixed lifestyle pattern`(생활권 혼합 리듬): 야간, 주말, 휴일 전날처럼 주거형 리듬을 나타내는 피처 축
- `location`(입지 피처): 표고, 버스정류장 수처럼 위치 특성을 나타내는 피처 축

목표:
- `cluster03`에서 경량 피처 조합이 `bike_change`에도 유지 가능한지 확인한다.


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
TRAIN_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_test_2025.csv'
OUTPUT_DIR = WORK_DIR / 'output/data'

TARGET_CLUSTER = 3
TARGET_STATION_GROUP = '생활권 혼합형'
RANDOM_STATE = 42


In [2]:
FULL_FEATURES = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h',
    'is_weekend', 'is_night_hour', 'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos',
    'commute_morning_flag', 'commute_evening_flag',
    'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
    'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]

FEATURE_SETS = {
    'full_36': FULL_FEATURES,
    'subset_a_mixed_lifestyle_core': [
        'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
        'lag_1h', 'lag_24h', 'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_6h',
        'is_weekend', 'is_night_hour', 'is_commute_hour', 'hour_sin', 'hour_cos',
        'commute_morning_flag', 'commute_evening_flag'
    ],
    'subset_b_mixed_lifestyle_weather': [
        'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
        'temperature', 'humidity', 'precipitation', 'wind_speed', 'is_rainy',
        'lag_1h', 'lag_24h', 'rolling_mean_24h', 'rolling_mean_6h',
        'is_weekend', 'is_night_hour', 'hour_sin', 'hour_cos'
    ],
    'subset_c_mixed_lifestyle_location': [
        'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
        'lag_1h', 'lag_24h', 'rolling_mean_24h', 'rolling_mean_6h',
        'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
        'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
        'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
    ],
    'subset_d_current_compact': [
        'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
        'temperature', 'precipitation', 'lag_1h', 'lag_24h', 'rolling_mean_24h',
        'rolling_std_24h', 'rolling_mean_6h', 'is_weekend', 'is_night_hour',
        'hour_sin', 'hour_cos', 'subway_distance_m', 'restaurant_count_300m',
        'bus_stop_count_300m', 'cafe_count_300m', 'is_rainy'
    ]
}


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_group = train_raw.loc[train_raw['cluster'] == TARGET_CLUSTER].copy()
test_group = test_raw.loc[test_raw['cluster'] == TARGET_CLUSTER].copy()

train_2023 = train_group.loc[train_group['date'] < '2024-01-01'].copy()
valid_2024 = train_group.loc[train_group['date'] >= '2024-01-01'].copy()
test_2025 = test_group.copy()
full_train = train_group.copy()


In [4]:
BASE_CATEGORICAL = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_xy(df: pd.DataFrame, feature_cols: list[str]):
    X = df[feature_cols].copy()
    for col in [c for c in feature_cols if c in BASE_CATEGORICAL]:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def build_lgbm_model() -> LGBMRegressor:
    return LGBMRegressor(
        objective='regression',
        learning_rate=0.05,
        n_estimators=400,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        verbose=-1,
    )


In [5]:
results = []

for feature_set_name, feature_cols in FEATURE_SETS.items():
    X_train, y_train = prepare_xy(train_2023, feature_cols)
    X_valid, y_valid = prepare_xy(valid_2024, feature_cols)
    X_full, y_full = prepare_xy(full_train, feature_cols)
    X_test, y_test = prepare_xy(test_2025, feature_cols)

    cat_cols = [c for c in feature_cols if c in BASE_CATEGORICAL]
    model = build_lgbm_model()
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_2023', y_train, model.predict(X_train)))
    results.append(evaluate_predictions(feature_set_name, 'validation_2024', y_valid, model.predict(X_valid)))

    model = build_lgbm_model()
    model.fit(X_full, y_full, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_full_refit', y_full, model.predict(X_full)))
    results.append(evaluate_predictions(feature_set_name, 'test_2025_refit', y_test, model.predict(X_test)))

results_df = pd.DataFrame(results)
results_df['cluster'] = TARGET_CLUSTER
results_df['station_group'] = TARGET_STATION_GROUP
results_df['feature_count'] = results_df['model'].map({k: len(v) for k, v in FEATURE_SETS.items()})

metrics_path = OUTPUT_DIR / 'ddri_bike_change_cluster03_second_round_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
results_df


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_cluster03_second_round_metrics.csv


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,train_2023,0.6601,0.4648,0.6659,0.9026,3,생활권 혼합형,35
1,full_36,validation_2024,0.7977,0.5533,0.4601,0.8878,3,생활권 혼합형,35
2,full_36,train_full_refit,0.7015,0.4888,0.6034,0.8950,3,생활권 혼합형,35
3,full_36,test_2025_refit,0.6542,0.4554,0.4671,0.9156,3,생활권 혼합형,35
4,subset_a_mixed_lifestyle_core,train_2023,0.6975,0.4928,0.6270,0.9053,3,생활권 혼합형,19
5,subset_a_mixed_lifestyle_core,validation_2024,0.8082,0.5583,0.4458,0.8931,3,생활권 혼합형,19
6,subset_a_mixed_lifestyle_core,train_full_refit,0.7302,0.5090,0.5703,0.8985,3,생활권 혼합형,19
7,subset_a_mixed_lifestyle_core,test_2025_refit,0.6745,0.4792,0.4335,0.9250,3,생활권 혼합형,19
8,subset_b_mixed_lifestyle_weather,train_2023,0.6763,0.4748,0.6493,0.9009,3,생활권 혼합형,20
9,subset_b_mixed_lifestyle_weather,validation_2024,0.8040,0.5594,0.4515,0.8899,3,생활권 혼합형,20


In [6]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,test_2025_refit,0.6542,0.4554,0.4671,0.9156,3,생활권 혼합형,35
1,subset_d_current_compact,test_2025_refit,0.6717,0.4786,0.4382,0.9208,3,생활권 혼합형,23
2,subset_b_mixed_lifestyle_weather,test_2025_refit,0.6733,0.4789,0.4356,0.9161,3,생활권 혼합형,20
3,subset_c_mixed_lifestyle_location,test_2025_refit,0.6733,0.4764,0.4355,0.9206,3,생활권 혼합형,19
4,subset_a_mixed_lifestyle_core,test_2025_refit,0.6745,0.4792,0.4335,0.9250,3,생활권 혼합형,19
